# 00_DATA_FEASIBILITY_v2_1

**Trương Minh Hoàng · MSSV 25410008**

Notebook nghiên cứu KLTN v2.1. Đọc [hướng dẫn chạy](README.md) và chuẩn bị dữ liệu trước khi thực thi. Output Colab cũ được bỏ; kết quả đã khóa nằm ở [evidence/historical](../evidence/historical/).

Research notebook: dataset-backed execution is separate from the small public demonstrations. Scientific feature formulas, model candidates and routing checks are retained. Public packaging changes cover paths, optional installation and fresh-run defaults.

# 00 — Cổng dữ liệu và khả năng thực thi (v2.1)

Notebook này chỉ kiểm tra dữ liệu/môi trường và tạo cấu hình dùng chung. **Không đánh giá RQ1 trên test**.

- LendingClub: `MyDrive/KLTN/data/LendingClub/LC_loans_granting_model_dataset.csv`
- DGraphFin: `MyDrive/KLTN/data/DGraphFin/raw/DGraphFin.zip`
- Chạy phát triển: `FINAL_MODE = False`, `FINAL_TEST_AUTHORISED = False`.
- Sau khi notebook 02 v2.1 tạo `rq1_freeze_manifest.json` và đã kiểm tra manifest, mới đặt cả hai cờ `True` để chạy final RQ1.

**Ranh giới bắt buộc:** notebook 00 không đọc phân bố nhãn của `test_mask`; test labels chỉ được truy cập trong nhánh final của notebook 02 v2.1 sau gate.


In [ ]:
# Install the optional research requirements before running.
# %pip -q install "torch-geometric>=2.5,<3" "lightgbm>=4.3,<5" pyarrow psutil scipy scikit-learn

In [ ]:
import os
from pathlib import Path
import hashlib, json, os, platform, sys, time
import numpy as np
import pandas as pd
import psutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Không chạy trong Colab; sửa PROJECT_ROOT nếu cần.')

# === Chỉ sửa PROJECT_ROOT và hai cờ final tại đây ===
PROJECT_ROOT = Path(os.environ.get('KLTN_WORKSPACE', '/content/drive/MyDrive/KLTN_REPRODUCTION_25410008' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'kltn_workspace'))).resolve()
FINAL_MODE = False
FINAL_TEST_AUTHORISED = False
TECHNICAL_RERUN_OVERRIDE = False

PROTOCOL_VERSION = 'klt_protocol_v2.1'
PRIMARY_SEED = 42
STABILITY_SEEDS = [7, 21, 84, 2026]
GRAPH_VISIBILITY_POLICY = 'official_transductive_unlabeled_topology_and_node_features'

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'protocol_v2_1'
CREDIT_DIR = ARTIFACT_DIR / 'credit'
RQ1_DIR = ARTIFACT_DIR / 'rq1'
RQ2_DIR = ARTIFACT_DIR / 'rq2'
CACHE_DIR = ARTIFACT_DIR / 'cache'
LC_CSV = DATA_DIR / 'LendingClub' / 'LC_loans_granting_model_dataset.csv'
DGRAPH_ROOT = DATA_DIR / 'DGraphFin'
DGRAPH_ZIP = DGRAPH_ROOT / 'raw' / 'DGraphFin.zip'
CONFIG_PATH = ARTIFACT_DIR / 'run_config.json'
POINTER_PATH = Path(os.environ.get('KLTN_CONFIG_POINTER', '/content/drive/MyDrive/KLTN_REPRODUCTION_CONFIG_POINTER.json' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'KLTN_REPRODUCTION_CONFIG_POINTER.json'))).resolve()

for directory in [DATA_DIR, ARTIFACT_DIR, CREDIT_DIR, RQ1_DIR, RQ2_DIR, CACHE_DIR, LC_CSV.parent, DGRAPH_ZIP.parent]:
    directory.mkdir(parents=True, exist_ok=True)

config = {
    'protocol_version': PROTOCOL_VERSION,
    'project_root': str(PROJECT_ROOT),
    'data_dir': str(DATA_DIR),
    'artifact_dir': str(ARTIFACT_DIR),
    'credit_dir': str(CREDIT_DIR),
    'rq1_dir': str(RQ1_DIR),
    'rq2_dir': str(RQ2_DIR),
    'cache_dir': str(CACHE_DIR),
    'lendingclub_csv': str(LC_CSV),
    'dgraph_root': str(DGRAPH_ROOT),
    'dgraph_zip': str(DGRAPH_ZIP),
    'final_mode': FINAL_MODE,
    'final_test_authorised': FINAL_TEST_AUTHORISED,
    'technical_rerun_override': TECHNICAL_RERUN_OVERRIDE,
    'primary_seed': PRIMARY_SEED,
    'stability_seeds': STABILITY_SEEDS,
    'graph_visibility_policy': GRAPH_VISIBILITY_POLICY,
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
POINTER_PATH.write_text(json.dumps({'config_path': str(CONFIG_PATH)}, indent=2), encoding='utf-8')
print(json.dumps(config, ensure_ascii=False, indent=2))


In [ ]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def file_record(path: Path, with_hash: bool = False) -> dict:
    record = {'path': str(path), 'exists': path.exists()}
    if path.exists():
        record['size_bytes'] = int(path.stat().st_size)
        if with_hash:
            record['sha256'] = sha256_file(path)
    return record

def mask_to_index(mask) -> np.ndarray:
    values = mask.cpu().numpy() if hasattr(mask, 'cpu') else np.asarray(mask)
    values = values.reshape(-1)
    return np.flatnonzero(values) if values.dtype == bool else values.astype('int64')

report = {
    'protocol_version': PROTOCOL_VERSION,
    'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'environment': {
        'python': sys.version,
        'platform': platform.platform(),
        'ram_total_gb': round(psutil.virtual_memory().total / 2**30, 2),
        'ram_available_gb': round(psutil.virtual_memory().available / 2**30, 2),
        'disk_free_gb': round(psutil.disk_usage('/content' if Path('/content').exists() else '/').free / 2**30, 2),
    },
    'files': {
        'lendingclub_csv': file_record(LC_CSV, with_hash=FINAL_MODE),
        'dgraph_zip': file_record(DGRAPH_ZIP, with_hash=FINAL_MODE),
    },
    'rq1_test_labels_inspected': False,
}
report


In [ ]:
from pathlib import Path

LC = LC_CSV

print("Size MB:", LC.stat().st_size / 1024**2)

with open(LC, "rb") as f:
    magic = f.read(8)

print("Magic bytes:", magic)

In [ ]:
import gzip
import shutil
from pathlib import Path

LC = LC_CSV

GZ_BACKUP = LC.with_name("LC_loans_granting_model_dataset.download.gz")
TMP = LC.with_name("LC_loans_granting_model_dataset.tmp.csv")

with open(LC, "rb") as f:
    magic = f.read(2)

if magic == b"\x1f\x8b":
    if not GZ_BACKUP.exists():
        shutil.copy2(LC, GZ_BACKUP)

    with gzip.open(LC, "rb") as src, open(TMP, "wb") as dst:
        shutil.copyfileobj(src, dst)

    TMP.replace(LC)
    print("Đã giải nén thành CSV thật.")
else:
    print("Không phải gzip, không sửa file.")

print("Final size MB:", LC.stat().st_size / 1024**2)

In [ ]:
import pandas as pd

df = pd.read_csv(LC, nrows=5, low_memory=False)

print(df.columns.tolist())
display(df)

In [ ]:
required_credit_columns = {'id', 'issue_d', 'Default'}
credit_audit = {'ready': False}

if LC_CSV.exists():
    header = pd.read_csv(LC_CSV, nrows=0)
    sample = pd.read_csv(LC_CSV, nrows=5000, low_memory=False)
    missing = sorted(required_credit_columns - set(header.columns))
    parsed_dates = pd.to_datetime(sample['issue_d'], errors='coerce') if 'issue_d' in sample else pd.Series(dtype='datetime64[ns]')
    target_values = sorted(pd.to_numeric(sample['Default'], errors='coerce').dropna().unique().tolist()) if 'Default' in sample else []
    credit_audit = {
        'ready': (not missing) and bool(parsed_dates.notna().all()) and set(target_values).issubset({0, 1}),
        'n_columns': int(len(header.columns)),
        'missing_required_columns': missing,
        'sample_rows': int(len(sample)),
        'sample_issue_d_parse_rate': float(parsed_dates.notna().mean()) if len(sample) else None,
        'sample_target_values': target_values,
    }
else:
    credit_audit['reason'] = 'Chưa có tệp LendingClub tại đường dẫn cấu hình.'

report['lendingclub'] = credit_audit
print(json.dumps(credit_audit, ensure_ascii=False, indent=2))


In [ ]:
dgraph_audit = {'ready': False}

if not DGRAPH_ZIP.exists():
    dgraph_audit['reason'] = 'Chưa có DGraphFin.zip tại thư mục raw.'
else:
    started = time.time()
    rss_before = psutil.Process().memory_info().rss
    try:
        import torch
        import torch_geometric
        from torch_geometric.datasets import DGraphFin

        dataset = DGraphFin(root=str(DGRAPH_ROOT))
        data = dataset[0]
        required = ['x', 'y', 'edge_index', 'train_mask', 'val_mask', 'test_mask']
        missing = [name for name in required if not hasattr(data, name)]
        if missing:
            raise ValueError(f'Thiếu đối tượng DGraphFin bắt buộc: {missing}')

        n_nodes = int(data.x.shape[0])
        n_edges = int(data.edge_index.shape[1])
        train_idx = mask_to_index(data.train_mask)
        val_idx = mask_to_index(data.val_mask)
        test_idx = mask_to_index(data.test_mask)  # chỉ đọc chỉ mục, KHÔNG đọc y[test_idx]

        masks_disjoint = (
            np.intersect1d(train_idx, val_idx, assume_unique=False).size == 0 and
            np.intersect1d(train_idx, test_idx, assume_unique=False).size == 0 and
            np.intersect1d(val_idx, test_idx, assume_unique=False).size == 0
        )
        index_bounds_ok = all(
            (len(idx) == 0) or (int(idx.min()) >= 0 and int(idx.max()) < n_nodes)
            for idx in [train_idx, val_idx, test_idx]
        )
        edge_min = int(data.edge_index.min()) if n_edges else 0
        edge_max = int(data.edge_index.max()) if n_edges else -1
        edge_bounds_ok = (n_edges == 0) or (edge_min >= 0 and edge_max < n_nodes)

        # Chỉ train/validation được dùng để kiểm tra class feasibility trước freeze.
        train_y = data.y[torch.as_tensor(train_idx, dtype=torch.long)].cpu().numpy().reshape(-1)
        val_y = data.y[torch.as_tensor(val_idx, dtype=torch.long)].cpu().numpy().reshape(-1)
        train_labeled = train_y[np.isin(train_y, [0, 1])]
        val_labeled = val_y[np.isin(val_y, [0, 1])]
        train_counts = {str(int(k)): int(v) for k, v in zip(*np.unique(train_labeled, return_counts=True))}
        val_counts = {str(int(k)): int(v) for k, v in zip(*np.unique(val_labeled, return_counts=True))}
        train_two_classes = set(np.unique(train_labeled).tolist()) == {0, 1}
        val_two_classes = set(np.unique(val_labeled).tolist()) == {0, 1}

        edge_type_present = hasattr(data, 'edge_type')
        edge_time_present = hasattr(data, 'edge_time')
        edge_type_shape = list(data.edge_type.shape) if edge_type_present else None
        edge_time_shape = list(data.edge_time.shape) if edge_time_present else None
        edge_aux_shapes_ok = (
            (not edge_type_present or int(data.edge_type.numel()) == n_edges) and
            (not edge_time_present or int(data.edge_time.numel()) == n_edges)
        )

        dgraph_audit = {
            'ready': bool(masks_disjoint and index_bounds_ok and edge_bounds_ok and edge_aux_shapes_ok and edge_type_present and train_two_classes and val_two_classes),
            'torch_version': torch.__version__,
            'torch_geometric_version': torch_geometric.__version__,
            'n_nodes': n_nodes,
            'n_edges': n_edges,
            'x_shape': list(data.x.shape),
            'y_shape': list(data.y.shape),
            'train_mask_count': int(len(train_idx)),
            'validation_mask_count': int(len(val_idx)),
            'test_mask_count': int(len(test_idx)),
            'train_labeled_01_counts': train_counts,
            'validation_labeled_01_counts': val_counts,
            'test_label_distribution_inspected': False,
            'masks_disjoint': bool(masks_disjoint),
            'index_bounds_ok': bool(index_bounds_ok),
            'edge_bounds_ok': bool(edge_bounds_ok),
            'edge_type_present': bool(edge_type_present),
            'edge_type_shape': edge_type_shape,
            'edge_time_present': bool(edge_time_present),
            'edge_time_shape': edge_time_shape,
            'edge_aux_shapes_ok': bool(edge_aux_shapes_ok),
            'graph_visibility_policy': GRAPH_VISIBILITY_POLICY,
            'current_rq1_requires_edge_type': True,
            'official_shape_reference_only': {'n_nodes': 3700550, 'n_edges': 4300999, 'x_dim': 17},
            'official_shape_matches': bool(n_nodes == 3700550 and n_edges == 4300999 and int(data.x.shape[1]) == 17),
            'load_seconds': round(time.time() - started, 2),
            'rss_increase_gb': round((psutil.Process().memory_info().rss - rss_before) / 2**30, 3),
        }
        del train_y, val_y, train_labeled, val_labeled, train_idx, val_idx, test_idx, data, dataset
    except Exception as exc:
        dgraph_audit = {'ready': False, 'error_type': type(exc).__name__, 'reason': str(exc)}

report['dgraphfin'] = dgraph_audit
print(json.dumps(dgraph_audit, ensure_ascii=False, indent=2))


In [ ]:
report['overall_ready'] = bool(
    report.get('lendingclub', {}).get('ready') and report.get('dgraphfin', {}).get('ready')
)
output_path = ARTIFACT_DIR / 'feasibility_report.json'
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Đã lưu: {output_path}')
print('KẾT LUẬN:', 'PASS — có thể tiếp tục' if report['overall_ready'] else 'CHƯA PASS — xem lý do ở trên')
if FINAL_MODE and not FINAL_TEST_AUTHORISED:
    print('LƯU Ý: FINAL_MODE=True nhưng FINAL_TEST_AUTHORISED=False; notebook 02 sẽ KHÔNG mở test.')
